<a href="https://colab.research.google.com/github/Daudi-Zacharia/msc-thesis/blob/main/segmentation/clean_notebooks/segmentation_unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lung Segmentation with a VGG16-BN U-Net

Clean rebuild of the **UNET 2** pipeline (PART 5 of `Segmentation_Unet_Full.ipynb`).
This is the segmentation model actually used by every classification experiment in
this repository: it produces the lung masks behind the "segmented" schemes in
`d1_experiments.ipynb` and `d2_experiments.ipynb`.

Superseded experiments — UNET 1, UNET 4, and the earlier `best_seg_project_*`
runs — are deliberately excluded. They remain in
`segmentation/raw_notebooks/Segmentation_Unet_Full.ipynb`.

**Data.** D1 (704 chest X-rays) is the only dataset with ground-truth lung masks,
so it is the only one the U-Net can be trained on. Split 80/10/10.

**Checkpoint.** Trained weights are saved to and loaded from Google Drive at

```
/content/drive/MyDrive/MSc Data Science/Thesis/Codes/segmentation/seg_unet_2.pth
```

Checkpoints are **not** stored in this repository. In the raw notebook the same
weights were written twice under two names (`best_seg_project_unet_improve_2.pth`
during training, then re-saved as `seg_unet_2.pth`); this notebook standardises on
`seg_unet_2.pth`, the name every classification notebook loads.

Requires a GPU runtime. The training section is kept for reproducibility but is not
meant to be re-run every time — the evaluation section loads the checkpoint instead.

# 1. Setup

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install torchmetrics -q
!pip install kagglehub -q

In [ ]:
import os
import random
import shutil
import time

import numpy as np
import PIL
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import tv_tensors
from torchvision.transforms import v2 as T

# reproducibility
SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# where the trained U-Net lives (Drive, not this repo)
path_model = '/content/drive/MyDrive/MSc Data Science/Thesis/Codes/segmentation'
SEG_CHECKPOINT = os.path.join(path_model, 'seg_unet_2.pth')

IMG_SIZE = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 2. Helper Functions

Everything reusable is defined once here: the loss, the training loop, the
history plot, the prediction pass and the qualitative viewer.

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice loss on the lung class only.

    Background dominates a chest X-ray mask, so scoring it would make the loss
    look good while the lungs stay poorly segmented.
    """

    def forward(self, inputs, targets, smooth=1):
        # inputs: (B, C, H, W) raw logits | targets: (B, H, W) class indices
        inputs = F.softmax(inputs, dim=1)

        targets_one_hot = F.one_hot(targets, num_classes=inputs.shape[1])
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        # class index 1 = lung
        input_flat = inputs[:, 1, :, :].contiguous().view(-1)
        target_flat = targets_one_hot[:, 1, :, :].contiguous().view(-1)

        intersection = (input_flat * target_flat).sum()
        dice = (2. * intersection + smooth) / (input_flat.sum() + target_flat.sum() + smooth)

        return 1 - dice

In [ ]:
def train_unet(model, train_dl, val_dl, optimizer, loss_fn_ce, save_path,
               epochs=15, patience=5, device='cpu'):
    """Trains with CrossEntropy + Dice, an LR scheduler and early stopping.

    The best-validation-loss state dict is checkpointed to save_path.
    """
    model.to(device)
    loss_fn_dice = DiceLoss().to(device)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    history = {'Training': {'Loss': [], 'Pixel Accuracy': [], 'Dice Score': []},
               'Validation': {'Loss': [], 'Pixel Accuracy': [], 'Dice Score': []}}

    best_val_loss = np.inf
    counter = 0

    for epoch in range(1, epochs + 1):

        # --- train ---
        model.train()
        train_loss = train_dice_sum = 0
        correct_px = total_px = 0

        for batch, target in train_dl:
            inputs, target = batch.to(device), target.to(device).long()
            output = model(inputs)

            dice_loss = loss_fn_dice(output, target)
            loss = loss_fn_ce(output, target) + dice_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_dice_sum += 1 - dice_loss.item()   # Dice score = 1 - Dice loss
            correct_px += (output.argmax(dim=1) == target).sum().item()
            total_px += target.numel()

        history['Training']['Loss'].append(train_loss / len(train_dl))
        history['Training']['Dice Score'].append(train_dice_sum / len(train_dl))
        history['Training']['Pixel Accuracy'].append(correct_px / total_px)

        # --- validate ---
        model.eval()
        val_loss = val_dice_sum = 0
        correct_px = total_px = 0

        with torch.no_grad():
            for batch, target in val_dl:
                inputs, target = batch.to(device), target.to(device).long()
                output = model(inputs)

                dice_loss = loss_fn_dice(output, target)
                val_loss += (loss_fn_ce(output, target) + dice_loss).item()
                val_dice_sum += 1 - dice_loss.item()
                correct_px += (output.argmax(dim=1) == target).sum().item()
                total_px += target.numel()

        val_loss_avg = val_loss / len(val_dl)
        val_dice_avg = val_dice_sum / len(val_dl)
        val_acc = correct_px / total_px

        scheduler.step(val_loss_avg)

        history['Validation']['Loss'].append(val_loss_avg)
        history['Validation']['Dice Score'].append(val_dice_avg)
        history['Validation']['Pixel Accuracy'].append(val_acc)

        # --- checkpoint + early stopping ---
        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            torch.save(model.state_dict(), save_path)
            print(f'Saved model at epoch {epoch} with Val Loss: {val_loss_avg:.4f}')
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f'Early stopping triggered at epoch {epoch}')
            break

        print(f"Epoch [{epoch}/{epochs}]  "
              f"train loss {history['Training']['Loss'][-1]:.4f}  "
              f"dice {history['Training']['Dice Score'][-1]:.4f}  |  "
              f"val loss {val_loss_avg:.4f}  acc {val_acc:.4f}  dice {val_dice_avg:.4f}")

    return history


def plot_history(history):
    """Training vs validation curves for each tracked metric."""
    metrics = ['Loss', 'Pixel Accuracy', 'Dice Score']
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))

    x = range(1, len(history['Training']['Loss']) + 1)
    for ax, metric in zip(axes, metrics):
        ax.plot(x, history['Training'][metric], label=f'Training {metric}')
        ax.plot(x, history['Validation'][metric], label=f'Validation {metric}')
        ax.set_xlabel('Epochs')
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def predict_segmentation(model, load_path, test_dl, device='cpu'):
    """Loads a checkpoint into `model` and predicts masks for the whole test set.

    Returns (predictions, targets, pixel_accuracy) with the two mask tensors
    shaped (N, H, W) as class indices.
    """
    state_dict = torch.load(load_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    correct_px = total_px = 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch, targets in test_dl:
            inputs, targets = batch.to(device), targets.to(device).long()
            preds = model(inputs).argmax(dim=1)

            correct_px += (preds == targets).sum().item()
            total_px += targets.numel()

            all_preds.append(preds.cpu())
            all_targets.append(targets.cpu())

    accuracy = correct_px / total_px
    print(f'Test set pixel accuracy: {accuracy:.4f}')

    return torch.cat(all_preds), torch.cat(all_targets), accuracy


def show_predictions(model, dataset, n=3, device='cpu'):
    """Original / ground truth / prediction / error map for the first n images."""
    model.to(device)
    model.eval()

    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i in range(n):
        img, mask_true = dataset[i]

        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu()

        # undo the ImageNet normalization for display
        img_shown = (img.cpu() * torch.tensor(IMAGENET_STD).view(3, 1, 1)
                     + torch.tensor(IMAGENET_MEAN).view(3, 1, 1)).clamp(0, 1)

        panels = [(img_shown.permute(1, 2, 0).numpy(), 'Original', None),
                  (mask_true.numpy(), 'True mask', 'gray'),
                  (pred.numpy(), 'Predicted mask', 'gray'),
                  ((pred != mask_true).float().numpy(), 'Error (pred != true)', 'gray')]

        for ax, (data, title, cmap) in zip(axes[i], panels):
            ax.imshow(data, cmap=cmap, **({} if cmap is None else dict(vmin=0, vmax=1)))
            ax.set_title(title)
            ax.axis('off')

    plt.tight_layout()
    plt.show()

# 3. Model Definition

In [ ]:
from torchvision.models import vgg16_bn


def conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


def up_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """U-Net with a pretrained VGG16-BN encoder and batch-normed up-convolutions."""

    def __init__(self, num_classes=2):
        super().__init__()

        self.encoder = vgg16_bn(weights='DEFAULT').features

        self.block1 = nn.Sequential(*self.encoder[:6])
        self.block2 = nn.Sequential(*self.encoder[6:13])
        self.block3 = nn.Sequential(*self.encoder[13:20])
        self.block4 = nn.Sequential(*self.encoder[20:27])
        self.block5 = nn.Sequential(*self.encoder[27:34])

        self.bottleneck = nn.Sequential(*self.encoder[34:])
        self.conv_bottleneck = conv(512, 1024)   # widen before upsampling

        self.up_conv6 = up_conv(1024, 512)
        self.conv6 = conv(512 + 512, 512)
        self.up_conv7 = up_conv(512, 256)
        self.conv7 = conv(256 + 512, 256)
        self.up_conv8 = up_conv(256, 128)
        self.conv8 = conv(128 + 256, 128)
        self.up_conv9 = up_conv(128, 64)
        self.conv9 = conv(64 + 128, 64)
        self.up_conv10 = up_conv(64, 32)
        self.conv10 = conv(32 + 64, 32)

        self.conv11 = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # encoder, keeping each stage for the skip connections
        block1 = self.block1(x)          # (64, 224, 224)
        block2 = self.block2(block1)     # (128, 112, 112)
        block3 = self.block3(block2)     # (256, 56, 56)
        block4 = self.block4(block3)     # (512, 28, 28)
        block5 = self.block5(block4)     # (512, 14, 14)

        x = self.conv_bottleneck(self.bottleneck(block5))   # (1024, 7, 7)

        # decoder: upsample, concatenate the matching encoder stage, convolve
        x = self.conv6(torch.cat([self.up_conv6(x), block5], dim=1))
        x = self.conv7(torch.cat([self.up_conv7(x), block4], dim=1))
        x = self.conv8(torch.cat([self.up_conv8(x), block3], dim=1))
        x = self.conv9(torch.cat([self.up_conv9(x), block2], dim=1))
        x = self.conv10(torch.cat([self.up_conv10(x), block1], dim=1))

        return self.conv11(x)            # (num_classes, 224, 224)

# 4. Data (D1, 704 images with masks)

Downloads the dataset and splits it 80/10/10 into `train/val/test`, each holding
an `images/` and a `masks/` folder.

In [ ]:
import kagglehub

path_704 = kagglehub.dataset_download("iamtapendu/chest-x-ray-lungs-segmentation")
print("Path to dataset files:", path_704)

In [ ]:
def database_setup(root_dir):
    """Splits the 704 images and their masks 80/10/10 into train/val/test."""

    im_path = os.path.join(root_dir, 'Chest-X-Ray', 'Chest-X-Ray', 'image')
    mask_path = os.path.join(root_dir, 'Chest-X-Ray', 'Chest-X-Ray', 'mask')

    for folder in ['train', 'val', 'test']:
        for subfolder in ['images', 'masks']:
            os.makedirs(os.path.join(folder, subfolder), exist_ok=True)

    # skip hidden files that would otherwise desynchronise image/mask pairing
    im_ls = sorted(f for f in os.listdir(im_path)
                   if f.endswith('png') and not f.startswith('.'))
    mask_ls = sorted(f for f in os.listdir(mask_path)
                     if f.endswith('png') and not f.startswith('.'))

    print(f'Images found: {len(im_ls)}')
    print(f'Masks found: {len(mask_ls)}')

    size = len(im_ls)
    train_size = int(size * 0.8)
    val_size = int(size * 0.1)
    val_end = train_size + val_size + 1

    splits = {
        'train': (im_ls[:train_size], mask_ls[:train_size]),
        'val': (im_ls[train_size:val_end], mask_ls[train_size:val_end]),
        'test': (im_ls[val_end:], mask_ls[val_end:]),
    }

    print('Dividing dataset into directories...')
    for split, (imgs, masks) in splits.items():
        for img in imgs:
            shutil.copy(os.path.join(im_path, img), os.path.join(split, 'images'))
        for mask in masks:
            shutil.copy(os.path.join(mask_path, mask), os.path.join(split, 'masks'))
        print(f'  {split}: {len(imgs)} images')

    return im_ls, mask_ls


im_ls, mask_ls = database_setup(path_704)

In [ ]:
class CustomDataset(Dataset):
    """Chest X-ray + lung mask pairs.

    Geometric augmentations are applied jointly to image and mask (via tv_tensors,
    which keeps them in step); colour jitter and normalization touch the image only.
    Validation and test use `val=True`, i.e. resize and normalize with no augmentation.
    """

    def __init__(self, root_dir, val=False):
        self.val = val

        self.joint_transforms = T.Compose([
            T.Resize(IMG_SIZE),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomAffine(degrees=10,          # patients are not always straight
                           translate=(0.1, 0.1),
                           scale=(0.9, 1.1)),   # controls apparent lung size
        ])

        self.im_transforms = T.Compose([
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

        self.val_im_transforms = T.Compose([
            T.Resize(IMG_SIZE),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

        self.im_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')

        self.im_ls = sorted(os.listdir(self.im_dir))
        self.mask_ls = sorted(os.listdir(self.mask_dir))

    def __len__(self):
        return len(self.im_ls)

    def __getitem__(self, idx):
        image = PIL.Image.open(os.path.join(self.im_dir, self.im_ls[idx])).convert('RGB')
        mask = PIL.Image.open(os.path.join(self.mask_dir, self.mask_ls[idx]))

        image = tv_tensors.Image(image)
        mask = tv_tensors.Mask(mask)

        if self.val:
            image = self.val_im_transforms(image)
            mask = T.Resize(IMG_SIZE)(mask)
        else:
            image, mask = self.joint_transforms(image, mask)
            image = self.im_transforms(image)

        # masks are stored as 0/255; CrossEntropy needs 0/1 class indices
        mask = (mask / 255).long()
        mask = mask.squeeze(0)      # (1, H, W) -> (H, W)

        return image, mask


train_td = CustomDataset('train')
val_td = CustomDataset('val', val=True)
test_td = CustomDataset('test', val=True)

train_dl = DataLoader(train_td, batch_size=32, shuffle=True)
val_dl = DataLoader(val_td, batch_size=38)     # val and test hold 70 images each
test_dl = DataLoader(test_td, batch_size=38)

print(f'train: {len(train_td)} | val: {len(val_td)} | test: {len(test_td)}')

image, mask = train_td[0]
print(f'image {tuple(image.shape)} {image.dtype} | mask {tuple(mask.shape)} '
      f'{mask.dtype} | mask values {mask.unique().tolist()}')

# 5. Training

Kept for reproducibility; it is not meant to be re-run on every pass. Skip to
section 6, which loads the saved checkpoint from Drive.

In [ ]:
model = UNet(num_classes=2)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn_ce = nn.CrossEntropyLoss()

history = train_unet(model, train_dl, val_dl, optimizer, loss_fn_ce,
                     save_path=SEG_CHECKPOINT,
                     epochs=15, patience=5, device=device)

plot_history(history)

# 6. Test-Set Evaluation

Loads `seg_unet_2.pth` from Drive and scores it on the held-out test split.

**On the Dice/IoU numbers.** `torchmetrics.segmentation.DiceScore` defaults to
`input_format='one-hot'`, but `predict_segmentation` returns class-index tensors
shaped `(N, H, W)`. Fed the wrong format it silently misreads the spatial
dimension as a class dimension and returns a meaningless value — which is how
Dice once came out *below* IoU, something that cannot happen for the same
predictions since Dice >= IoU always. Passing `input_format='index'` fixes it.
Both `micro` and `macro` averages are reported so Dice and IoU are always
compared on matched terms.

In [ ]:
model = UNet(num_classes=2)

preds, targets, acc = predict_segmentation(model, SEG_CHECKPOINT, test_dl, device=device)

In [ ]:
from torchmetrics import JaccardIndex
from torchmetrics.segmentation import DiceScore

print(f'Pixel accuracy: {acc:.4f}\n')

for avg in ['micro', 'macro']:
    # input_format='index' matches the (N, H, W) class-index tensors above
    dice = DiceScore(num_classes=2, average=avg, input_format='index')
    jaccard = JaccardIndex(task='multiclass', num_classes=2, average=avg)

    print(f"average='{avg}'  ->  Dice: {dice(preds, targets).item():.4f}   "
          f"IoU: {jaccard(preds, targets).item():.4f}")

## Qualitative check

In [ ]:
show_predictions(model, test_td, n=3, device=device)